In [ ]:
#library for ML framework 
import tensorflow as tf
"""
Sequential is the way to build model with the tf framework, we
can add layers, build, compile and fit the complete model 
"""
from tensorflow.keras.models import Sequential 
"""
Embedding layer - the bridge from events (represented as integers via IDs) and make
them vectors with fixed size - https://www.tensorflow.org/api_docs/python/tf/keras/layers/Embedding
how to represent events to computers?
We can have a fixed amount of events, each one represented with vector full of zeros except one slot of the specific event 
this is called one-hot encoding the shortcoming of this method is the lack of context between similar events on the system. 
in addition, the data is very sparse but still in this method spend a lot of space.
with Embedding layer we represent an event with a vector full of real numbers (in most cases the vector has low dimensions)
during training this layer can learn when events occur in similar conditions and then the points on the vector space are become close to each other. 
In this project the dimension of each vector is 16 and we have 175 evetns (matrix 175X16)
each row is a specific event (like hashtable) this vector is inputed to the LSTM layer 

LSTM layer - Long Short Term Memory (Hochreiter 1997) represented the LSTM model that I covered in the article 
https://www.tensorflow.org/api_docs/python/tf/keras/layers/LSTM

Dense Layer - this is the normal NN connection when each neuron connected to each one of the next layer
we can choose the activation function (linear on defualt)
in this case for exaple the output layer is like this and we get vacab_size outputs (one for each event) 
and output the unnormalize probability to each event to be the next 

"""
from tensorflow.keras.models import Embedding, LSTM, Dense


def build_ladohd_model(vocab_size=175, embedding_dim=16, hidden_size=64, seq_length=64):
    """
    Builds the LADOHD LSTM anomaly detection model.
    
    Args:
        vocab_size (int): Total number of unique system events(fixed number).
        embedding_dim (int): Dimension of the dense embedding vectors.
        hidden_size (int): Number of features in the LSTM hidden state.
        seq_length (int): Length of the input event sequences (BPTT window).
        
    Returns:
        tf.keras.Model: The compiled sequential model.
    """
    model = Sequential([
        # 1. Embedding Layer: Converts categorical event IDs into dense vectors
        Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=seq_length),
        
        # 2. LSTM Layers: 3 stacked layers to learn complex temporal patterns
        # return_sequences=True ensures the entire sequence is passed to the next layer
        LSTM(hidden_size, return_sequences=True),
        LSTM(hidden_size, return_sequences=True),
        LSTM(hidden_size, return_sequences=True),
        
        # 3. Fully Connected Layer: Extracts non-linear features from the LSTM outputs
        Dense(100, activation='relu'),
        
        # 4. Output Layer: Unnormalized log probabilities (logits) for each event in the vocabulary
        Dense(vocab_size)
    ])
    
    return model

if __name__ == "__main__":
    model = build_ladohd_model()
    
    # Compile the model with Adam optimizer and Sparse Categorical Crossentropy loss
    model.compile(
        optimizer='adam',
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=['accuracy']
    )
    
    model.summary()


In [ ]:
import pandas as pd 
import numpy as np 
def load_and_filter_logs(file_path,target_actor="powershell.exe") -> pd.DataFrame:
    """
    Read raw EDR logs (CSV format) and filters them by specific actor.
    
    Args:
        file_path (str): path to the log file.
        target_actors (str, optional): The process name to monitor. Defaults to "powershell.exe".
    Returns:
        pd.DataFrame: A filtered DataFrame ordered by time. 
    """
    #read the log file (in csv format)
    df = pd.read_csv(file_path)
    #sort the logs by the time they occured
    df = df.sort_values(by='Timestamp')
    #take just the rows that the actor is the one we filtering (and make a copy to not change the original)
    filtered_df = df[df['Actor'] == target_actor].copy()
    return filtered_df

def build_event_vocabulary(df,feature_coulmns): 
    """
    Implements the transformation function F_T(.) to map raw features into a categorical vocabulary.
    
    Args:
        df (pd.DataFrame): The filtered logs.
        feature_columns (list): List of columns to define a unique event (e.g , ['EventType', 'Action', 'Target'] (coulmn names)).
        
    Returns:
        tuple: (List of sequential event IDs, Dictionary mapping feature tuples to IDs)
    """
    #for each combination of those coulmns make unique event 
    unique_events = df[feature_coulmns].drop_duplicates()
    #map each event to unique ID 
    vocab = {tuple(row) : idx for idx,row in enumerate(unique_events.values)}
    #make all the rows in the logs to sequence of ID's (continuous list) (the df is sorted by Timestamps)
    event_sequence  = [vocab[tuple(x)] for x in df[feature_coulmns].values]
    return event_sequence,vocab
def creating_training_sequence(event_sequence,seq_length=64): 
    """
    Generates 3D tensors for LSTM training using a sliding window approach.
    
    Args:
        event_sequence (list): The full chronologial sequence of event IDs.
        seq_length (int): The BPTT unrolling window size (default: 64).
        
    Returns:
        tuple: (X_train numpy array, y_train numpy array)
    """
    X,y = [],[] 
    #make sliding windows (each windows of events is with a length of seq_length )
    #the output is the next event each time (we want to give probability to the next event in real time)
    #if the event is with normal probability(top K probable events) (according to the connection the model learns-the training data is this windows) 
    #this is benign action else this is anomalous action 
    for i in range(len(event_sequence) - seq_length):
        X.append(event_sequence[i:i+seq_length])
        y.append(event_sequence[i+seq_length])
    return np.array(X),np.array(y)


In [ ]:
#"Stop training when a monitored metric has stopped improving." - https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/EarlyStopping
from tensorflow.keras.callbacks import EarlyStopping


def train_ladohd_model(model, X_train, y_train, epochs=50, batch_size=64, validation_split=0.2):
    """
    Trains the LSTM model using the prepared sequences of benign events.
    
    Args:
        model: The compiled Keras sequential model.
        X_train (np.array): Input sequences (Sliding windows).
        y_train (np.array): Target next-events.
        epochs (int): Maximum number of training iterations.
        batch_size (int): Number of sequences to process before updating weights.
        validation_split (float): Fraction of data to use for validation.
        
    Returns:
        History object containing training metrics.
    """
    #define this process of earlyStopping to prevent overfitting 
    early_stopping = EarlyStopping(
        monitor='val_loss', 
        patience=5,          
        restore_best_weights=True
    )
    #see https://www.tensorflow.org/api_docs/python/tf/keras/Model#fit
    #X_train and y_train are the data to train on 
    #epochs is the number of iteration over the whole dataset, the training can be stop before the number of epochs is the maximum (early_stopping)
    #batch_size is the number of samples that the model reads in one time (the mini-batch I explained)
    #validation_split takes the friction given from the dataset and uses it to validates the model (and not for training)
    #callbacks are list of function that performed during the trainings
    history =  model.fit(
        X_train,
        y_train,
        epochs=epochs,
        batch_size = batch_size, 
        validation_split= validation_split, 
        callbacks= [early_stopping], 
        verbose = 1
    )
    return history